In [ ]:
# =========================================
# Phase 5 - Segmentation (Time-based Windowing)
# Builds per-window metadata aligned to Phase 4 episodes
# Output: phase_05_windows/{subject}__windows.parquet
# Last update: 2026-02-12
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2B_DIR     = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_NORM_DIR = PROCESSED_ROOT / "phase_03_normalization" / "normalized_env_per_subject"
PHASE4_DIR      = PROCESSED_ROOT / "phase_04_events"
PHASE5_DIR      = PROCESSED_ROOT / "phase_05_windows"

PHASE5_DIR.mkdir(parents=True, exist_ok=True)

print("Paths loaded.")
print("PHASE5_DIR:", PHASE5_DIR)

# ─── Config ────────────────────────────────────────
WIN_LEN_S = 0.200
OVERLAP = 0.50
STEP_S = WIN_LEN_S * (1.0 - OVERLAP)

# Window QC
MIN_COVERAGE_RATIO = 0.60       # >=60% of expected samples must exist
MAX_TIME_GAP_S = 0.050          # flag if max Δt inside window > 50ms
INCLUDE_CONTEXT = False         # default: windows strictly within [t_on, t_off]
CONTEXT_PAD_S = 0.30            # only if INCLUDE_CONTEXT=True

# Minimal plotting / QC
PLOT_QC = True
MAX_TRIALS_TO_PLOT = 6          # avoid huge plotting loops

print(f"Windowing: win={WIN_LEN_S:.3f}s | overlap={OVERLAP:.2f} | step={STEP_S:.3f}s")
print(f"QC: MIN_COVERAGE_RATIO={MIN_COVERAGE_RATIO} | MAX_TIME_GAP_S={MAX_TIME_GAP_S}s")
print(f"Context: INCLUDE_CONTEXT={INCLUDE_CONTEXT} | CONTEXT_PAD_S={CONTEXT_PAD_S}s")

# ─── Helpers ───────────────────────────────────────
def estimate_fs(t: np.ndarray) -> float:
    """Robust sampling rate estimate from time vector."""
    t = np.asarray(t, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 10:
        return np.nan
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 5:
        return np.nan
    # robust: use median dt
    return float(1.0 / np.median(dt))

def make_windows(t_on: float, t_off: float, win_len_s: float, step_s: float):
    """Windows fully inside [t_on, t_off]. Returns list of (t_start, t_end)."""
    if (t_off - t_on) < win_len_s:
        return []
    starts = np.arange(t_on, t_off - win_len_s + 1e-12, step_s)
    return [(float(s), float(s + win_len_s)) for s in starts]

def slice_by_time(df: pd.DataFrame, tcol: str, t_start: float, t_end: float) -> pd.DataFrame:
    """Inclusive start, exclusive end."""
    return df.loc[(df[tcol] >= t_start) & (df[tcol] < t_end)]

def window_timegap_ok(t: np.ndarray, max_gap_s: float) -> tuple[bool, float]:
    if len(t) < 2:
        return False, np.nan
    gaps = np.diff(t)
    gaps = gaps[np.isfinite(gaps)]
    if len(gaps) == 0:
        return False, np.nan
    mg = float(np.max(gaps))
    return (mg <= max_gap_s), mg

def expected_samples(fs: float, win_len_s: float) -> float:
    if not np.isfinite(fs) or fs <= 0:
        return np.nan
    return float(fs * win_len_s)

def qc_coverage(n_samples: int, n_expected: float, min_ratio: float) -> bool:
    if not np.isfinite(n_expected) or n_expected <= 0:
        # if fs unknown, accept based on absolute minimal samples (fallback)
        return n_samples >= 5
    return (n_samples >= min_ratio * n_expected)

# ─── QC Plot ───────────────────────────────────────
def plot_trial_windows(subject_name: str, trial_id: str, episodes_trial: pd.DataFrame, windows_trial: pd.DataFrame):
    """Plot gate (IMU if available else EMG gate proxy) and overlay episode and some windows."""
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    plt.figure(figsize=(14, 4))

    plotted = False

    # Try IMU gate first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) &
                        (df_imu["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_imu.empty and "t_imu" in df_imu.columns:
            df_imu = df_imu.sort_values("t_imu")
            # Use available gate column(s)
            if "gyro_gate" in df_imu.columns:
                plt.plot(df_imu["t_imu"], df_imu["gyro_gate"], lw=1.1, label="gyro_gate (IMU)")
                plotted = True

    # EMG fallback proxy: max(env_norm_*) if IMU not plotted
    if (not plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) &
                        (df_emg["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_emg.empty and "t_emg" in df_emg.columns:
            df_emg = df_emg.sort_values("t_emg")
            env_cols = [c for c in df_emg.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_emg[env_cols].max(axis=1)
                plt.plot(df_emg["t_emg"], gate, lw=1.1, label="max(env_norm_*) (EMG proxy)")
                plotted = True

    # Overlay episodes
    for i, r in episodes_trial.sort_values("t_on").iterrows():
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.12, color="orange", label="episode" if i == episodes_trial.index[0] else None)

    # Overlay a few windows (first 40)
    wshow = windows_trial.sort_values("t_start").head(40)
    for j, r in wshow.iterrows():
        plt.axvline(r["t_start"], alpha=0.25, lw=0.8, color="black")

    plt.title(f"{subject_name} — {trial_id} | episodes={len(episodes_trial)} | windows={len(windows_trial)}")
    plt.xlabel("Time (s)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Phase 5 runner ────────────────────────────────
def run_phase5_for_subject(subject_name: str):
    ep_path  = PHASE4_DIR / f"{subject_name}__episodes.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"

    if not ep_path.exists():
        raise FileNotFoundError(f"Episodes file not found: {ep_path}")

    episodes = pd.read_parquet(ep_path)
    episodes["subject"] = episodes["subject"].astype(str)
    episodes["trial_id"] = episodes["trial_id"].astype(str)

    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()
    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()

    if not df_emg.empty:
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
    if not df_imu.empty:
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)

    # Pre-estimate fs per trial (robust + fast)
    fs_emg_by_trial = {}
    fs_imu_by_trial = {}

    if not df_emg.empty and "t_emg" in df_emg.columns:
        for tid, g in df_emg.groupby("trial_id"):
            fs_emg_by_trial[tid] = estimate_fs(g["t_emg"].to_numpy())
    if not df_imu.empty and "t_imu" in df_imu.columns:
        for tid, g in df_imu.groupby("trial_id"):
            fs_imu_by_trial[tid] = estimate_fs(g["t_imu"].to_numpy())

    # Build windows
    rows = []

    for (trial_id, episode_id), ep_grp in episodes.groupby(["trial_id", "episode_id"]):
        ep0 = ep_grp.iloc[0]
        t_on  = float(ep0["t_on"])
        t_off = float(ep0["t_off"])

        # optional context
        if INCLUDE_CONTEXT:
            t_on_eff  = t_on  - CONTEXT_PAD_S
            t_off_eff = t_off + CONTEXT_PAD_S
        else:
            t_on_eff, t_off_eff = t_on, t_off

        win_list = make_windows(t_on_eff, t_off_eff, WIN_LEN_S, STEP_S)
        if not win_list:
            continue

        # trial signals (only once)
        emg_trial = pd.DataFrame()
        imu_trial = pd.DataFrame()

        if not df_emg.empty:
            emg_trial = df_emg[df_emg["trial_id"] == trial_id].sort_values("t_emg")
        if not df_imu.empty:
            imu_trial = df_imu[df_imu["trial_id"] == trial_id].sort_values("t_imu")

        fs_emg = fs_emg_by_trial.get(trial_id, np.nan)
        fs_imu = fs_imu_by_trial.get(trial_id, np.nan)
        exp_emg = expected_samples(fs_emg, WIN_LEN_S)
        exp_imu = expected_samples(fs_imu, WIN_LEN_S)

        for w_id, (ts, te) in enumerate(win_list, start=1):
            emg_win = slice_by_time(emg_trial, "t_emg", ts, te) if not emg_trial.empty else pd.DataFrame()
            imu_win = slice_by_time(imu_trial, "t_imu", ts, te) if not imu_trial.empty else pd.DataFrame()

            # --- counts (handle long-format) ---
            emg_n_rows = int(len(emg_win))
            imu_n_rows = int(len(imu_win))

            # unique time-samples (this is what "n samples" should mean)
            emg_n = int(emg_win["t_emg"].nunique()) if emg_n_rows > 0 else 0
            imu_n = int(imu_win["t_imu"].nunique()) if imu_n_rows > 0 else 0

            # coverage should use unique time sample counts
            emg_cov_ok = qc_coverage(emg_n, exp_emg, MIN_COVERAGE_RATIO) if emg_n > 0 else False
            imu_cov_ok = qc_coverage(imu_n, exp_imu, MIN_COVERAGE_RATIO) if imu_n > 0 else False

            # --- time-gap QC should use unique time vector (avoid repeated timestamps) ---
            emg_gap_ok, emg_max_gap = (False, np.nan)
            imu_gap_ok, imu_max_gap = (False, np.nan)

            if emg_n > 1:
                t_emg_u = np.sort(emg_win["t_emg"].unique())
                emg_gap_ok, emg_max_gap = window_timegap_ok(t_emg_u, MAX_TIME_GAP_S)

            if imu_n > 1:
                t_imu_u = np.sort(imu_win["t_imu"].unique())
                imu_gap_ok, imu_max_gap = window_timegap_ok(t_imu_u, MAX_TIME_GAP_S)

            emg_ok = bool(emg_cov_ok and emg_gap_ok)
            imu_ok = bool(imu_cov_ok and imu_gap_ok)


            rows.append({
                "subject": subject_name,
                "trial_id": trial_id,
                "episode_id": int(episode_id),
                "window_id": int(w_id),

                "t_start": float(ts),
                "t_end": float(te),
                "win_len_s": float(WIN_LEN_S),
                "step_s": float(STEP_S),

                "task_type": ep0.get("task_type", "unknown"),
                "data_source": ep0.get("data_source", "unknown"),
                "gate_source": ep0.get("gate_source", "unknown"),

                "t_on": t_on,
                "t_off": t_off,
                "context_used": bool(INCLUDE_CONTEXT),

                "fs_emg_est": float(fs_emg) if np.isfinite(fs_emg) else np.nan,
                "fs_imu_est": float(fs_imu) if np.isfinite(fs_imu) else np.nan,
                "emg_expected_n": float(exp_emg) if np.isfinite(exp_emg) else np.nan,
                "imu_expected_n": float(exp_imu) if np.isfinite(exp_imu) else np.nan,
                "emg_n": emg_n,              # unique time samples
                "imu_n": imu_n,              # unique time samples
                "emg_n_rows": emg_n_rows,    # raw rows (debug / long-format)
                "imu_n_rows": imu_n_rows,
                "emg_cov_ok": bool(emg_cov_ok),
                "imu_cov_ok": bool(imu_cov_ok),
                "emg_gap_ok": bool(emg_gap_ok),
                "imu_gap_ok": bool(imu_gap_ok),
                "emg_max_gap_s": float(emg_max_gap) if np.isfinite(emg_max_gap) else np.nan,
                "imu_max_gap_s": float(imu_max_gap) if np.isfinite(imu_max_gap) else np.nan,
                "emg_ok": bool(emg_ok),
                "imu_ok": bool(imu_ok),
            })

    windows_df = pd.DataFrame(rows)

    out_path = PHASE5_DIR / f"{subject_name}__windows.parquet"
    windows_df.to_parquet(out_path, index=False)

    print(f"\nPhase 5 completed for {subject_name}")
    print(f"  → windows: {out_path}")
    if windows_df.empty:
        print("  WARNING: windows_df is empty. Check episodes or win params.")
        return windows_df

    # Summary
    print("\nWindow QC summary (mean of booleans):")
    display(windows_df[["emg_cov_ok","emg_gap_ok","emg_ok","imu_cov_ok","imu_gap_ok","imu_ok"]].mean(numeric_only=True))

    print("\nWindows per trial (top 12):")
    display(
        windows_df.groupby(["trial_id","episode_id"])["window_id"].count()
                 .reset_index(name="n_windows")
                 .sort_values("n_windows", ascending=False)
                 .head(12)
    )

    # Optional QC plots
    if PLOT_QC:
        trials = windows_df["trial_id"].unique().tolist()
        trials = trials[:MAX_TRIALS_TO_PLOT]
        print(f"\nPlotting QC for {len(trials)} trials (max={MAX_TRIALS_TO_PLOT}) ...")
        for tid in trials:
            ep_t = episodes[episodes["trial_id"] == tid]
            w_t  = windows_df[windows_df["trial_id"] == tid]
            plot_trial_windows(subject_name, tid, ep_t, w_t)

    return windows_df




In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_1"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_2"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_3"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_4"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_5"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_6"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_7"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "Healthy_Subject_8"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_1"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_2"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_3"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_4"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_5"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_7"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_8"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_9"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_10"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_14"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_15"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_16"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)

In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# =========================================
# Phase 5 - Segmentation (Time-based Windowing) — UPDATED (modality-aware + long-format safe)
# Builds per-window metadata aligned to Phase 4 episodes
# Output: phase_05_windows/{subject}__windows.parquet
# Last update: 2026-02-15
#
# Key updates:
#  1) Handles missing IMU / missing EMG per-window:
#     - emg_ok / imu_ok become NaN when that modality is missing in the window (instead of False)
#     - adds has_emg / has_imu flags
#  2) Long-format safe counting:
#     - emg_n / imu_n = number of UNIQUE timestamps in the window (not number of rows)
#     - adds emg_n_rows / imu_n_rows for debugging
#  3) Time-gap QC uses UNIQUE timestamps (avoids repeated timestamps from long format)
#  4) rows.append stores NaN as-is (no bool(...) casting that would corrupt NaNs)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2B_DIR     = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_NORM_DIR = PROCESSED_ROOT / "phase_03_normalization" / "normalized_env_per_subject"
PHASE4_DIR      = PROCESSED_ROOT / "phase_04_events"
PHASE5_DIR      = PROCESSED_ROOT / "phase_05_windows"

PHASE5_DIR.mkdir(parents=True, exist_ok=True)

print("Paths loaded.")
print("PHASE5_DIR:", PHASE5_DIR)

# ─── Config ────────────────────────────────────────
WIN_LEN_S = 0.200
OVERLAP = 0.50
STEP_S = WIN_LEN_S * (1.0 - OVERLAP)

# Window QC
MIN_COVERAGE_RATIO = 0.60       # >=60% of expected unique timestamps must exist
MAX_TIME_GAP_S = 0.050          # flag if max Δt inside window > 50ms (on unique timestamps)
INCLUDE_CONTEXT = False         # default: windows strictly within [t_on, t_off]
CONTEXT_PAD_S = 0.30            # only if INCLUDE_CONTEXT=True

# Minimal plotting / QC
PLOT_QC = True
MAX_TRIALS_TO_PLOT = 6

print(f"Windowing: win={WIN_LEN_S:.3f}s | overlap={OVERLAP:.2f} | step={STEP_S:.3f}s")
print(f"QC: MIN_COVERAGE_RATIO={MIN_COVERAGE_RATIO} | MAX_TIME_GAP_S={MAX_TIME_GAP_S}s")
print(f"Context: INCLUDE_CONTEXT={INCLUDE_CONTEXT} | CONTEXT_PAD_S={CONTEXT_PAD_S}s")

# ─── Helpers ───────────────────────────────────────
def estimate_fs(t: np.ndarray) -> float:
    """Robust sampling rate estimate from time vector (seconds)."""
    t = np.asarray(t, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 10:
        return np.nan
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 5:
        return np.nan
    return float(1.0 / np.median(dt))

def make_windows(t_on: float, t_off: float, win_len_s: float, step_s: float):
    """Windows fully inside [t_on, t_off]. Returns list of (t_start, t_end)."""
    if (t_off - t_on) < win_len_s:
        return []
    starts = np.arange(t_on, t_off - win_len_s + 1e-12, step_s)
    return [(float(s), float(s + win_len_s)) for s in starts]

def slice_by_time(df: pd.DataFrame, tcol: str, t_start: float, t_end: float) -> pd.DataFrame:
    """Inclusive start, exclusive end."""
    return df.loc[(df[tcol] >= t_start) & (df[tcol] < t_end)]

def window_timegap_ok(t_unique_sorted: np.ndarray, max_gap_s: float) -> tuple[bool, float]:
    """Check max gap on UNIQUE timestamps."""
    t = np.asarray(t_unique_sorted, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 2:
        return False, np.nan
    gaps = np.diff(t)
    gaps = gaps[np.isfinite(gaps)]
    if len(gaps) == 0:
        return False, np.nan
    mg = float(np.max(gaps))
    return (mg <= max_gap_s), mg

def expected_samples(fs: float, win_len_s: float) -> float:
    if not np.isfinite(fs) or fs <= 0:
        return np.nan
    return float(fs * win_len_s)

def qc_coverage(n_samples_unique_time: int, n_expected: float, min_ratio: float) -> bool:
    """Coverage check using UNIQUE timestamp counts."""
    if not np.isfinite(n_expected) or n_expected <= 0:
        return n_samples_unique_time >= 5
    return (n_samples_unique_time >= min_ratio * n_expected)

# ─── QC Plot ───────────────────────────────────────
def plot_trial_windows(subject_name: str, trial_id: str, episodes_trial: pd.DataFrame, windows_trial: pd.DataFrame):
    """
    Plot IMU gyro_gate if available; otherwise plot EMG proxy (max env_norm_*).
    Overlay episodes and some window start lines.
    """
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    plt.figure(figsize=(14, 4))
    plotted = False

    # Try IMU gate first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) &
                        (df_imu["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_imu.empty and "t_imu" in df_imu.columns and "gyro_gate" in df_imu.columns:
            df_imu = df_imu.sort_values("t_imu")
            plt.plot(df_imu["t_imu"], df_imu["gyro_gate"], lw=1.1, label="gyro_gate (IMU)")
            plotted = True

    # EMG fallback proxy if IMU not plotted
    if (not plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) &
                        (df_emg["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_emg.empty and "t_emg" in df_emg.columns:
            df_emg = df_emg.sort_values("t_emg")
            env_cols = [c for c in df_emg.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_emg[env_cols].max(axis=1)
                plt.plot(df_emg["t_emg"], gate, lw=1.1, label="max(env_norm_*) (EMG proxy)")
                plotted = True

    # Overlay episodes
    episodes_trial = episodes_trial.sort_values("t_on")
    for i, (_, r) in enumerate(episodes_trial.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.12, color="orange", label="episode" if i == 0 else None)

    # Overlay a few window starts (downsample to avoid clutter)
    wshow = windows_trial.sort_values("t_start").head(60)
    for _, r in wshow.iterrows():
        plt.axvline(r["t_start"], alpha=0.18, lw=0.8, color="black")

    title_extra = "" if plotted else " (no gate plotted)"
    plt.title(f"{subject_name} — {trial_id} | episodes={len(episodes_trial)} | windows={len(windows_trial)}{title_extra}")
    plt.xlabel("Time (s)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Phase 5 runner ────────────────────────────────
def run_phase5_for_subject(subject_name: str):
    ep_path  = PHASE4_DIR / f"{subject_name}__episodes.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"

    if not ep_path.exists():
        raise FileNotFoundError(f"Episodes file not found: {ep_path}")

    episodes = pd.read_parquet(ep_path)
    episodes["subject"] = episodes["subject"].astype(str)
    episodes["trial_id"] = episodes["trial_id"].astype(str)

    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()
    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()

    if not df_emg.empty:
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
        if "t_emg" not in df_emg.columns:
            raise ValueError(f"EMG file missing 't_emg': {emg_path}")

    if not df_imu.empty:
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
        if "t_imu" not in df_imu.columns:
            raise ValueError(f"IMU file missing 't_imu': {imu_path}")

    # Pre-estimate fs per trial using UNIQUE timestamps (robust to long-format)
    fs_emg_by_trial = {}
    fs_imu_by_trial = {}

    if not df_emg.empty:
        for tid, g in df_emg.groupby("trial_id"):
            fs_emg_by_trial[tid] = estimate_fs(np.sort(g["t_emg"].unique()))

    if not df_imu.empty:
        for tid, g in df_imu.groupby("trial_id"):
            fs_imu_by_trial[tid] = estimate_fs(np.sort(g["t_imu"].unique()))

    rows = []

    # Build windows per (trial, episode)
    for (trial_id, episode_id), ep_grp in episodes.groupby(["trial_id", "episode_id"]):
        ep0 = ep_grp.iloc[0]
        t_on  = float(ep0["t_on"])
        t_off = float(ep0["t_off"])

        # optional context
        if INCLUDE_CONTEXT:
            t_on_eff  = t_on  - CONTEXT_PAD_S
            t_off_eff = t_off + CONTEXT_PAD_S
        else:
            t_on_eff, t_off_eff = t_on, t_off

        win_list = make_windows(t_on_eff, t_off_eff, WIN_LEN_S, STEP_S)
        if not win_list:
            continue

        # Trial data (may be missing modality)
        emg_trial = pd.DataFrame()
        imu_trial = pd.DataFrame()

        if not df_emg.empty:
            emg_trial = df_emg[df_emg["trial_id"] == trial_id].sort_values("t_emg")
        if not df_imu.empty:
            imu_trial = df_imu[df_imu["trial_id"] == trial_id].sort_values("t_imu")

        fs_emg = fs_emg_by_trial.get(trial_id, np.nan)
        fs_imu = fs_imu_by_trial.get(trial_id, np.nan)
        exp_emg = expected_samples(fs_emg, WIN_LEN_S)
        exp_imu = expected_samples(fs_imu, WIN_LEN_S)

        for w_id, (ts, te) in enumerate(win_list, start=1):
            # Slice windows (can be empty if modality missing)
            emg_win = slice_by_time(emg_trial, "t_emg", ts, te) if not emg_trial.empty else pd.DataFrame()
            imu_win = slice_by_time(imu_trial, "t_imu", ts, te) if not imu_trial.empty else pd.DataFrame()

            has_emg_win = (len(emg_win) > 0)
            has_imu_win = (len(imu_win) > 0)

            # -------------------------
            # EMG QC (modality-aware)
            # -------------------------
            emg_n_rows = int(len(emg_win))
            emg_n = int(emg_win["t_emg"].nunique()) if emg_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_emg_win:
                emg_cov_ok  = np.nan
                emg_gap_ok  = np.nan
                emg_ok      = np.nan
                emg_max_gap = np.nan
            else:
                emg_cov_ok = qc_coverage(emg_n, exp_emg, MIN_COVERAGE_RATIO) if emg_n > 0 else False

                if emg_n > 1:
                    t_emg_u = np.sort(emg_win["t_emg"].unique())
                    emg_gap_ok, emg_max_gap = window_timegap_ok(t_emg_u, MAX_TIME_GAP_S)
                else:
                    emg_gap_ok, emg_max_gap = (False, np.nan)

                emg_ok = bool(emg_cov_ok and emg_gap_ok)

            # -------------------------
            # IMU QC (modality-aware)
            # -------------------------
            imu_n_rows = int(len(imu_win))
            imu_n = int(imu_win["t_imu"].nunique()) if imu_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_imu_win:
                imu_cov_ok  = np.nan
                imu_gap_ok  = np.nan
                imu_ok      = np.nan
                imu_max_gap = np.nan
            else:
                imu_cov_ok = qc_coverage(imu_n, exp_imu, MIN_COVERAGE_RATIO) if imu_n > 0 else False

                if imu_n > 1:
                    t_imu_u = np.sort(imu_win["t_imu"].unique())
                    imu_gap_ok, imu_max_gap = window_timegap_ok(t_imu_u, MAX_TIME_GAP_S)
                else:
                    imu_gap_ok, imu_max_gap = (False, np.nan)

                imu_ok = bool(imu_cov_ok and imu_gap_ok)

            rows.append({
                "subject": subject_name,
                "trial_id": str(trial_id),
                "episode_id": int(episode_id),
                "window_id": int(w_id),

                "t_start": float(ts),
                "t_end": float(te),
                "win_len_s": float(WIN_LEN_S),
                "step_s": float(STEP_S),

                "task_type": ep0.get("task_type", "unknown"),
                "data_source": ep0.get("data_source", "unknown"),
                "gate_source": ep0.get("gate_source", "unknown"),

                "t_on": t_on,
                "t_off": t_off,
                "context_used": bool(INCLUDE_CONTEXT),

                "fs_emg_est": float(fs_emg) if np.isfinite(fs_emg) else np.nan,
                "fs_imu_est": float(fs_imu) if np.isfinite(fs_imu) else np.nan,
                "emg_expected_n": float(exp_emg) if np.isfinite(exp_emg) else np.nan,
                "imu_expected_n": float(exp_imu) if np.isfinite(exp_imu) else np.nan,

                # modality availability
                "has_emg": bool(has_emg_win),
                "has_imu": bool(has_imu_win),

                # unique sample counts (for QC / features)
                "emg_n": int(emg_n),
                "imu_n": int(imu_n),

                # raw row counts (debug for long-format)
                "emg_n_rows": int(emg_n_rows),
                "imu_n_rows": int(imu_n_rows),

                # QC flags (keep NaN as NaN; DO NOT cast to bool)
                "emg_cov_ok": emg_cov_ok,
                "imu_cov_ok": imu_cov_ok,
                "emg_gap_ok": emg_gap_ok,
                "imu_gap_ok": imu_gap_ok,
                "emg_max_gap_s": float(emg_max_gap) if np.isfinite(emg_max_gap) else np.nan,
                "imu_max_gap_s": float(imu_max_gap) if np.isfinite(imu_max_gap) else np.nan,
                "emg_ok": emg_ok,
                "imu_ok": imu_ok,
            })

    windows_df = pd.DataFrame(rows)

    out_path = PHASE5_DIR / f"{subject_name}__windows.parquet"
    windows_df.to_parquet(out_path, index=False)

    print(f"\nPhase 5 completed for {subject_name}")
    print(f"  → windows: {out_path}")
    if windows_df.empty:
        print("  WARNING: windows_df is empty. Check episodes or win params.")
        return windows_df

    # Summary (NaNs are ignored by mean — exactly what we want for missing modalities)
    qc_cols = ["emg_cov_ok","emg_gap_ok","emg_ok","imu_cov_ok","imu_gap_ok","imu_ok"]
    print("\nWindow QC summary (mean; NaNs ignored):")
    display(windows_df[qc_cols].mean(numeric_only=True))

    # Extra: missing modality ratios
    print("\nMissing modality ratios:")
    print("  EMG missing ratio:", float((~windows_df["has_emg"]).mean()))
    print("  IMU missing ratio:", float((~windows_df["has_imu"]).mean()))

    print("\nWindows per trial (top 12):")
    display(
        windows_df.groupby(["trial_id","episode_id"])["window_id"].count()
                 .reset_index(name="n_windows")
                 .sort_values("n_windows", ascending=False)
                 .head(12)
    )

    # Optional QC plots
    if PLOT_QC:
        trials = windows_df["trial_id"].unique().tolist()[:MAX_TRIALS_TO_PLOT]
        print(f"\nPlotting QC for {len(trials)} trials (max={MAX_TRIALS_TO_PLOT}) ...")
        for tid in trials:
            ep_t = episodes[episodes["trial_id"] == tid]
            w_t  = windows_df[windows_df["trial_id"] == tid]
            plot_trial_windows(subject_name, tid, ep_t, w_t)

    return windows_df

# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_3"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)


In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# =========================================
# Phase 5 - Segmentation (Time-based Windowing) — UPDATED (modality-aware + long-format safe)
# Builds per-window metadata aligned to Phase 4 episodes
# Output: phase_05_windows/{subject}__windows.parquet
# Last update: 2026-02-15
#
# Key updates:
#  1) Handles missing IMU / missing EMG per-window:
#     - emg_ok / imu_ok become NaN when that modality is missing in the window (instead of False)
#     - adds has_emg / has_imu flags
#  2) Long-format safe counting:
#     - emg_n / imu_n = number of UNIQUE timestamps in the window (not number of rows)
#     - adds emg_n_rows / imu_n_rows for debugging
#  3) Time-gap QC uses UNIQUE timestamps (avoids repeated timestamps from long format)
#  4) rows.append stores NaN as-is (no bool(...) casting that would corrupt NaNs)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2B_DIR     = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_NORM_DIR = PROCESSED_ROOT / "phase_03_normalization" / "normalized_env_per_subject"
PHASE4_DIR      = PROCESSED_ROOT / "phase_04_events"
PHASE5_DIR      = PROCESSED_ROOT / "phase_05_windows"

PHASE5_DIR.mkdir(parents=True, exist_ok=True)

print("Paths loaded.")
print("PHASE5_DIR:", PHASE5_DIR)

# ─── Config ────────────────────────────────────────
WIN_LEN_S = 0.200
OVERLAP = 0.50
STEP_S = WIN_LEN_S * (1.0 - OVERLAP)

# Window QC
MIN_COVERAGE_RATIO = 0.60       # >=60% of expected unique timestamps must exist
MAX_TIME_GAP_S = 0.050          # flag if max Δt inside window > 50ms (on unique timestamps)
INCLUDE_CONTEXT = False         # default: windows strictly within [t_on, t_off]
CONTEXT_PAD_S = 0.30            # only if INCLUDE_CONTEXT=True

# Minimal plotting / QC
PLOT_QC = True
MAX_TRIALS_TO_PLOT = 6

print(f"Windowing: win={WIN_LEN_S:.3f}s | overlap={OVERLAP:.2f} | step={STEP_S:.3f}s")
print(f"QC: MIN_COVERAGE_RATIO={MIN_COVERAGE_RATIO} | MAX_TIME_GAP_S={MAX_TIME_GAP_S}s")
print(f"Context: INCLUDE_CONTEXT={INCLUDE_CONTEXT} | CONTEXT_PAD_S={CONTEXT_PAD_S}s")

# ─── Helpers ───────────────────────────────────────
def estimate_fs(t: np.ndarray) -> float:
    """Robust sampling rate estimate from time vector (seconds)."""
    t = np.asarray(t, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 10:
        return np.nan
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 5:
        return np.nan
    return float(1.0 / np.median(dt))

def make_windows(t_on: float, t_off: float, win_len_s: float, step_s: float):
    """Windows fully inside [t_on, t_off]. Returns list of (t_start, t_end)."""
    if (t_off - t_on) < win_len_s:
        return []
    starts = np.arange(t_on, t_off - win_len_s + 1e-12, step_s)
    return [(float(s), float(s + win_len_s)) for s in starts]

def slice_by_time(df: pd.DataFrame, tcol: str, t_start: float, t_end: float) -> pd.DataFrame:
    """Inclusive start, exclusive end."""
    return df.loc[(df[tcol] >= t_start) & (df[tcol] < t_end)]

def window_timegap_ok(t_unique_sorted: np.ndarray, max_gap_s: float) -> tuple[bool, float]:
    """Check max gap on UNIQUE timestamps."""
    t = np.asarray(t_unique_sorted, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 2:
        return False, np.nan
    gaps = np.diff(t)
    gaps = gaps[np.isfinite(gaps)]
    if len(gaps) == 0:
        return False, np.nan
    mg = float(np.max(gaps))
    return (mg <= max_gap_s), mg

def expected_samples(fs: float, win_len_s: float) -> float:
    if not np.isfinite(fs) or fs <= 0:
        return np.nan
    return float(fs * win_len_s)

def qc_coverage(n_samples_unique_time: int, n_expected: float, min_ratio: float) -> bool:
    """Coverage check using UNIQUE timestamp counts."""
    if not np.isfinite(n_expected) or n_expected <= 0:
        return n_samples_unique_time >= 5
    return (n_samples_unique_time >= min_ratio * n_expected)

# ─── QC Plot ───────────────────────────────────────
def plot_trial_windows(subject_name: str, trial_id: str, episodes_trial: pd.DataFrame, windows_trial: pd.DataFrame):
    """
    Plot IMU gyro_gate if available; otherwise plot EMG proxy (max env_norm_*).
    Overlay episodes and some window start lines.
    """
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    plt.figure(figsize=(14, 4))
    plotted = False

    # Try IMU gate first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) &
                        (df_imu["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_imu.empty and "t_imu" in df_imu.columns and "gyro_gate" in df_imu.columns:
            df_imu = df_imu.sort_values("t_imu")
            plt.plot(df_imu["t_imu"], df_imu["gyro_gate"], lw=1.1, label="gyro_gate (IMU)")
            plotted = True

    # EMG fallback proxy if IMU not plotted
    if (not plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) &
                        (df_emg["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_emg.empty and "t_emg" in df_emg.columns:
            df_emg = df_emg.sort_values("t_emg")
            env_cols = [c for c in df_emg.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_emg[env_cols].max(axis=1)
                plt.plot(df_emg["t_emg"], gate, lw=1.1, label="max(env_norm_*) (EMG proxy)")
                plotted = True

    # Overlay episodes
    episodes_trial = episodes_trial.sort_values("t_on")
    for i, (_, r) in enumerate(episodes_trial.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.12, color="orange", label="episode" if i == 0 else None)

    # Overlay a few window starts (downsample to avoid clutter)
    wshow = windows_trial.sort_values("t_start").head(60)
    for _, r in wshow.iterrows():
        plt.axvline(r["t_start"], alpha=0.18, lw=0.8, color="black")

    title_extra = "" if plotted else " (no gate plotted)"
    plt.title(f"{subject_name} — {trial_id} | episodes={len(episodes_trial)} | windows={len(windows_trial)}{title_extra}")
    plt.xlabel("Time (s)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Phase 5 runner ────────────────────────────────
def run_phase5_for_subject(subject_name: str):
    ep_path  = PHASE4_DIR / f"{subject_name}__episodes.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"

    if not ep_path.exists():
        raise FileNotFoundError(f"Episodes file not found: {ep_path}")

    episodes = pd.read_parquet(ep_path)
    episodes["subject"] = episodes["subject"].astype(str)
    episodes["trial_id"] = episodes["trial_id"].astype(str)

    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()
    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()

    if not df_emg.empty:
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
        if "t_emg" not in df_emg.columns:
            raise ValueError(f"EMG file missing 't_emg': {emg_path}")

    if not df_imu.empty:
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
        if "t_imu" not in df_imu.columns:
            raise ValueError(f"IMU file missing 't_imu': {imu_path}")

    # Pre-estimate fs per trial using UNIQUE timestamps (robust to long-format)
    fs_emg_by_trial = {}
    fs_imu_by_trial = {}

    if not df_emg.empty:
        for tid, g in df_emg.groupby("trial_id"):
            fs_emg_by_trial[tid] = estimate_fs(np.sort(g["t_emg"].unique()))

    if not df_imu.empty:
        for tid, g in df_imu.groupby("trial_id"):
            fs_imu_by_trial[tid] = estimate_fs(np.sort(g["t_imu"].unique()))

    rows = []

    # Build windows per (trial, episode)
    for (trial_id, episode_id), ep_grp in episodes.groupby(["trial_id", "episode_id"]):
        ep0 = ep_grp.iloc[0]
        t_on  = float(ep0["t_on"])
        t_off = float(ep0["t_off"])

        # optional context
        if INCLUDE_CONTEXT:
            t_on_eff  = t_on  - CONTEXT_PAD_S
            t_off_eff = t_off + CONTEXT_PAD_S
        else:
            t_on_eff, t_off_eff = t_on, t_off

        win_list = make_windows(t_on_eff, t_off_eff, WIN_LEN_S, STEP_S)
        if not win_list:
            continue

        # Trial data (may be missing modality)
        emg_trial = pd.DataFrame()
        imu_trial = pd.DataFrame()

        if not df_emg.empty:
            emg_trial = df_emg[df_emg["trial_id"] == trial_id].sort_values("t_emg")
        if not df_imu.empty:
            imu_trial = df_imu[df_imu["trial_id"] == trial_id].sort_values("t_imu")

        fs_emg = fs_emg_by_trial.get(trial_id, np.nan)
        fs_imu = fs_imu_by_trial.get(trial_id, np.nan)
        exp_emg = expected_samples(fs_emg, WIN_LEN_S)
        exp_imu = expected_samples(fs_imu, WIN_LEN_S)

        for w_id, (ts, te) in enumerate(win_list, start=1):
            # Slice windows (can be empty if modality missing)
            emg_win = slice_by_time(emg_trial, "t_emg", ts, te) if not emg_trial.empty else pd.DataFrame()
            imu_win = slice_by_time(imu_trial, "t_imu", ts, te) if not imu_trial.empty else pd.DataFrame()

            has_emg_win = (len(emg_win) > 0)
            has_imu_win = (len(imu_win) > 0)

            # -------------------------
            # EMG QC (modality-aware)
            # -------------------------
            emg_n_rows = int(len(emg_win))
            emg_n = int(emg_win["t_emg"].nunique()) if emg_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_emg_win:
                emg_cov_ok  = np.nan
                emg_gap_ok  = np.nan
                emg_ok      = np.nan
                emg_max_gap = np.nan
            else:
                emg_cov_ok = qc_coverage(emg_n, exp_emg, MIN_COVERAGE_RATIO) if emg_n > 0 else False

                if emg_n > 1:
                    t_emg_u = np.sort(emg_win["t_emg"].unique())
                    emg_gap_ok, emg_max_gap = window_timegap_ok(t_emg_u, MAX_TIME_GAP_S)
                else:
                    emg_gap_ok, emg_max_gap = (False, np.nan)

                emg_ok = bool(emg_cov_ok and emg_gap_ok)

            # -------------------------
            # IMU QC (modality-aware)
            # -------------------------
            imu_n_rows = int(len(imu_win))
            imu_n = int(imu_win["t_imu"].nunique()) if imu_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_imu_win:
                imu_cov_ok  = np.nan
                imu_gap_ok  = np.nan
                imu_ok      = np.nan
                imu_max_gap = np.nan
            else:
                imu_cov_ok = qc_coverage(imu_n, exp_imu, MIN_COVERAGE_RATIO) if imu_n > 0 else False

                if imu_n > 1:
                    t_imu_u = np.sort(imu_win["t_imu"].unique())
                    imu_gap_ok, imu_max_gap = window_timegap_ok(t_imu_u, MAX_TIME_GAP_S)
                else:
                    imu_gap_ok, imu_max_gap = (False, np.nan)

                imu_ok = bool(imu_cov_ok and imu_gap_ok)

            rows.append({
                "subject": subject_name,
                "trial_id": str(trial_id),
                "episode_id": int(episode_id),
                "window_id": int(w_id),

                "t_start": float(ts),
                "t_end": float(te),
                "win_len_s": float(WIN_LEN_S),
                "step_s": float(STEP_S),

                "task_type": ep0.get("task_type", "unknown"),
                "data_source": ep0.get("data_source", "unknown"),
                "gate_source": ep0.get("gate_source", "unknown"),

                "t_on": t_on,
                "t_off": t_off,
                "context_used": bool(INCLUDE_CONTEXT),

                "fs_emg_est": float(fs_emg) if np.isfinite(fs_emg) else np.nan,
                "fs_imu_est": float(fs_imu) if np.isfinite(fs_imu) else np.nan,
                "emg_expected_n": float(exp_emg) if np.isfinite(exp_emg) else np.nan,
                "imu_expected_n": float(exp_imu) if np.isfinite(exp_imu) else np.nan,

                # modality availability
                "has_emg": bool(has_emg_win),
                "has_imu": bool(has_imu_win),

                # unique sample counts (for QC / features)
                "emg_n": int(emg_n),
                "imu_n": int(imu_n),

                # raw row counts (debug for long-format)
                "emg_n_rows": int(emg_n_rows),
                "imu_n_rows": int(imu_n_rows),

                # QC flags (keep NaN as NaN; DO NOT cast to bool)
                "emg_cov_ok": emg_cov_ok,
                "imu_cov_ok": imu_cov_ok,
                "emg_gap_ok": emg_gap_ok,
                "imu_gap_ok": imu_gap_ok,
                "emg_max_gap_s": float(emg_max_gap) if np.isfinite(emg_max_gap) else np.nan,
                "imu_max_gap_s": float(imu_max_gap) if np.isfinite(imu_max_gap) else np.nan,
                "emg_ok": emg_ok,
                "imu_ok": imu_ok,
            })

    windows_df = pd.DataFrame(rows)

    out_path = PHASE5_DIR / f"{subject_name}__windows.parquet"
    windows_df.to_parquet(out_path, index=False)

    print(f"\nPhase 5 completed for {subject_name}")
    print(f"  → windows: {out_path}")
    if windows_df.empty:
        print("  WARNING: windows_df is empty. Check episodes or win params.")
        return windows_df

    # Summary (NaNs are ignored by mean — exactly what we want for missing modalities)
    qc_cols = ["emg_cov_ok","emg_gap_ok","emg_ok","imu_cov_ok","imu_gap_ok","imu_ok"]
    print("\nWindow QC summary (mean; NaNs ignored):")
    display(windows_df[qc_cols].mean(numeric_only=True))

    # Extra: missing modality ratios
    print("\nMissing modality ratios:")
    print("  EMG missing ratio:", float((~windows_df["has_emg"]).mean()))
    print("  IMU missing ratio:", float((~windows_df["has_imu"]).mean()))

    print("\nWindows per trial (top 12):")
    display(
        windows_df.groupby(["trial_id","episode_id"])["window_id"].count()
                 .reset_index(name="n_windows")
                 .sort_values("n_windows", ascending=False)
                 .head(12)
    )

    # Optional QC plots
    if PLOT_QC:
        trials = windows_df["trial_id"].unique().tolist()[:MAX_TRIALS_TO_PLOT]
        print(f"\nPlotting QC for {len(trials)} trials (max={MAX_TRIALS_TO_PLOT}) ...")
        for tid in trials:
            ep_t = episodes[episodes["trial_id"] == tid]
            w_t  = windows_df[windows_df["trial_id"] == tid]
            plot_trial_windows(subject_name, tid, ep_t, w_t)

    return windows_df

# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_11"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)


In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# =========================================
# Phase 5 - Segmentation (Time-based Windowing) — UPDATED (modality-aware + long-format safe)
# Builds per-window metadata aligned to Phase 4 episodes
# Output: phase_05_windows/{subject}__windows.parquet
# Last update: 2026-02-15
#
# Key updates:
#  1) Handles missing IMU / missing EMG per-window:
#     - emg_ok / imu_ok become NaN when that modality is missing in the window (instead of False)
#     - adds has_emg / has_imu flags
#  2) Long-format safe counting:
#     - emg_n / imu_n = number of UNIQUE timestamps in the window (not number of rows)
#     - adds emg_n_rows / imu_n_rows for debugging
#  3) Time-gap QC uses UNIQUE timestamps (avoids repeated timestamps from long format)
#  4) rows.append stores NaN as-is (no bool(...) casting that would corrupt NaNs)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2B_DIR     = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_NORM_DIR = PROCESSED_ROOT / "phase_03_normalization" / "normalized_env_per_subject"
PHASE4_DIR      = PROCESSED_ROOT / "phase_04_events"
PHASE5_DIR      = PROCESSED_ROOT / "phase_05_windows"

PHASE5_DIR.mkdir(parents=True, exist_ok=True)

print("Paths loaded.")
print("PHASE5_DIR:", PHASE5_DIR)

# ─── Config ────────────────────────────────────────
WIN_LEN_S = 0.200
OVERLAP = 0.50
STEP_S = WIN_LEN_S * (1.0 - OVERLAP)

# Window QC
MIN_COVERAGE_RATIO = 0.60       # >=60% of expected unique timestamps must exist
MAX_TIME_GAP_S = 0.050          # flag if max Δt inside window > 50ms (on unique timestamps)
INCLUDE_CONTEXT = False         # default: windows strictly within [t_on, t_off]
CONTEXT_PAD_S = 0.30            # only if INCLUDE_CONTEXT=True

# Minimal plotting / QC
PLOT_QC = True
MAX_TRIALS_TO_PLOT = 6

print(f"Windowing: win={WIN_LEN_S:.3f}s | overlap={OVERLAP:.2f} | step={STEP_S:.3f}s")
print(f"QC: MIN_COVERAGE_RATIO={MIN_COVERAGE_RATIO} | MAX_TIME_GAP_S={MAX_TIME_GAP_S}s")
print(f"Context: INCLUDE_CONTEXT={INCLUDE_CONTEXT} | CONTEXT_PAD_S={CONTEXT_PAD_S}s")

# ─── Helpers ───────────────────────────────────────
def estimate_fs(t: np.ndarray) -> float:
    """Robust sampling rate estimate from time vector (seconds)."""
    t = np.asarray(t, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 10:
        return np.nan
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 5:
        return np.nan
    return float(1.0 / np.median(dt))

def make_windows(t_on: float, t_off: float, win_len_s: float, step_s: float):
    """Windows fully inside [t_on, t_off]. Returns list of (t_start, t_end)."""
    if (t_off - t_on) < win_len_s:
        return []
    starts = np.arange(t_on, t_off - win_len_s + 1e-12, step_s)
    return [(float(s), float(s + win_len_s)) for s in starts]

def slice_by_time(df: pd.DataFrame, tcol: str, t_start: float, t_end: float) -> pd.DataFrame:
    """Inclusive start, exclusive end."""
    return df.loc[(df[tcol] >= t_start) & (df[tcol] < t_end)]

def window_timegap_ok(t_unique_sorted: np.ndarray, max_gap_s: float) -> tuple[bool, float]:
    """Check max gap on UNIQUE timestamps."""
    t = np.asarray(t_unique_sorted, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 2:
        return False, np.nan
    gaps = np.diff(t)
    gaps = gaps[np.isfinite(gaps)]
    if len(gaps) == 0:
        return False, np.nan
    mg = float(np.max(gaps))
    return (mg <= max_gap_s), mg

def expected_samples(fs: float, win_len_s: float) -> float:
    if not np.isfinite(fs) or fs <= 0:
        return np.nan
    return float(fs * win_len_s)

def qc_coverage(n_samples_unique_time: int, n_expected: float, min_ratio: float) -> bool:
    """Coverage check using UNIQUE timestamp counts."""
    if not np.isfinite(n_expected) or n_expected <= 0:
        return n_samples_unique_time >= 5
    return (n_samples_unique_time >= min_ratio * n_expected)

# ─── QC Plot ───────────────────────────────────────
def plot_trial_windows(subject_name: str, trial_id: str, episodes_trial: pd.DataFrame, windows_trial: pd.DataFrame):
    """
    Plot IMU gyro_gate if available; otherwise plot EMG proxy (max env_norm_*).
    Overlay episodes and some window start lines.
    """
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    plt.figure(figsize=(14, 4))
    plotted = False

    # Try IMU gate first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) &
                        (df_imu["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_imu.empty and "t_imu" in df_imu.columns and "gyro_gate" in df_imu.columns:
            df_imu = df_imu.sort_values("t_imu")
            plt.plot(df_imu["t_imu"], df_imu["gyro_gate"], lw=1.1, label="gyro_gate (IMU)")
            plotted = True

    # EMG fallback proxy if IMU not plotted
    if (not plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) &
                        (df_emg["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_emg.empty and "t_emg" in df_emg.columns:
            df_emg = df_emg.sort_values("t_emg")
            env_cols = [c for c in df_emg.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_emg[env_cols].max(axis=1)
                plt.plot(df_emg["t_emg"], gate, lw=1.1, label="max(env_norm_*) (EMG proxy)")
                plotted = True

    # Overlay episodes
    episodes_trial = episodes_trial.sort_values("t_on")
    for i, (_, r) in enumerate(episodes_trial.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.12, color="orange", label="episode" if i == 0 else None)

    # Overlay a few window starts (downsample to avoid clutter)
    wshow = windows_trial.sort_values("t_start").head(60)
    for _, r in wshow.iterrows():
        plt.axvline(r["t_start"], alpha=0.18, lw=0.8, color="black")

    title_extra = "" if plotted else " (no gate plotted)"
    plt.title(f"{subject_name} — {trial_id} | episodes={len(episodes_trial)} | windows={len(windows_trial)}{title_extra}")
    plt.xlabel("Time (s)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Phase 5 runner ────────────────────────────────
def run_phase5_for_subject(subject_name: str):
    ep_path  = PHASE4_DIR / f"{subject_name}__episodes.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"

    if not ep_path.exists():
        raise FileNotFoundError(f"Episodes file not found: {ep_path}")

    episodes = pd.read_parquet(ep_path)
    episodes["subject"] = episodes["subject"].astype(str)
    episodes["trial_id"] = episodes["trial_id"].astype(str)

    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()
    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()

    if not df_emg.empty:
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
        if "t_emg" not in df_emg.columns:
            raise ValueError(f"EMG file missing 't_emg': {emg_path}")

    if not df_imu.empty:
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
        if "t_imu" not in df_imu.columns:
            raise ValueError(f"IMU file missing 't_imu': {imu_path}")

    # Pre-estimate fs per trial using UNIQUE timestamps (robust to long-format)
    fs_emg_by_trial = {}
    fs_imu_by_trial = {}

    if not df_emg.empty:
        for tid, g in df_emg.groupby("trial_id"):
            fs_emg_by_trial[tid] = estimate_fs(np.sort(g["t_emg"].unique()))

    if not df_imu.empty:
        for tid, g in df_imu.groupby("trial_id"):
            fs_imu_by_trial[tid] = estimate_fs(np.sort(g["t_imu"].unique()))

    rows = []

    # Build windows per (trial, episode)
    for (trial_id, episode_id), ep_grp in episodes.groupby(["trial_id", "episode_id"]):
        ep0 = ep_grp.iloc[0]
        t_on  = float(ep0["t_on"])
        t_off = float(ep0["t_off"])

        # optional context
        if INCLUDE_CONTEXT:
            t_on_eff  = t_on  - CONTEXT_PAD_S
            t_off_eff = t_off + CONTEXT_PAD_S
        else:
            t_on_eff, t_off_eff = t_on, t_off

        win_list = make_windows(t_on_eff, t_off_eff, WIN_LEN_S, STEP_S)
        if not win_list:
            continue

        # Trial data (may be missing modality)
        emg_trial = pd.DataFrame()
        imu_trial = pd.DataFrame()

        if not df_emg.empty:
            emg_trial = df_emg[df_emg["trial_id"] == trial_id].sort_values("t_emg")
        if not df_imu.empty:
            imu_trial = df_imu[df_imu["trial_id"] == trial_id].sort_values("t_imu")

        fs_emg = fs_emg_by_trial.get(trial_id, np.nan)
        fs_imu = fs_imu_by_trial.get(trial_id, np.nan)
        exp_emg = expected_samples(fs_emg, WIN_LEN_S)
        exp_imu = expected_samples(fs_imu, WIN_LEN_S)

        for w_id, (ts, te) in enumerate(win_list, start=1):
            # Slice windows (can be empty if modality missing)
            emg_win = slice_by_time(emg_trial, "t_emg", ts, te) if not emg_trial.empty else pd.DataFrame()
            imu_win = slice_by_time(imu_trial, "t_imu", ts, te) if not imu_trial.empty else pd.DataFrame()

            has_emg_win = (len(emg_win) > 0)
            has_imu_win = (len(imu_win) > 0)

            # -------------------------
            # EMG QC (modality-aware)
            # -------------------------
            emg_n_rows = int(len(emg_win))
            emg_n = int(emg_win["t_emg"].nunique()) if emg_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_emg_win:
                emg_cov_ok  = np.nan
                emg_gap_ok  = np.nan
                emg_ok      = np.nan
                emg_max_gap = np.nan
            else:
                emg_cov_ok = qc_coverage(emg_n, exp_emg, MIN_COVERAGE_RATIO) if emg_n > 0 else False

                if emg_n > 1:
                    t_emg_u = np.sort(emg_win["t_emg"].unique())
                    emg_gap_ok, emg_max_gap = window_timegap_ok(t_emg_u, MAX_TIME_GAP_S)
                else:
                    emg_gap_ok, emg_max_gap = (False, np.nan)

                emg_ok = bool(emg_cov_ok and emg_gap_ok)

            # -------------------------
            # IMU QC (modality-aware)
            # -------------------------
            imu_n_rows = int(len(imu_win))
            imu_n = int(imu_win["t_imu"].nunique()) if imu_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_imu_win:
                imu_cov_ok  = np.nan
                imu_gap_ok  = np.nan
                imu_ok      = np.nan
                imu_max_gap = np.nan
            else:
                imu_cov_ok = qc_coverage(imu_n, exp_imu, MIN_COVERAGE_RATIO) if imu_n > 0 else False

                if imu_n > 1:
                    t_imu_u = np.sort(imu_win["t_imu"].unique())
                    imu_gap_ok, imu_max_gap = window_timegap_ok(t_imu_u, MAX_TIME_GAP_S)
                else:
                    imu_gap_ok, imu_max_gap = (False, np.nan)

                imu_ok = bool(imu_cov_ok and imu_gap_ok)

            rows.append({
                "subject": subject_name,
                "trial_id": str(trial_id),
                "episode_id": int(episode_id),
                "window_id": int(w_id),

                "t_start": float(ts),
                "t_end": float(te),
                "win_len_s": float(WIN_LEN_S),
                "step_s": float(STEP_S),

                "task_type": ep0.get("task_type", "unknown"),
                "data_source": ep0.get("data_source", "unknown"),
                "gate_source": ep0.get("gate_source", "unknown"),

                "t_on": t_on,
                "t_off": t_off,
                "context_used": bool(INCLUDE_CONTEXT),

                "fs_emg_est": float(fs_emg) if np.isfinite(fs_emg) else np.nan,
                "fs_imu_est": float(fs_imu) if np.isfinite(fs_imu) else np.nan,
                "emg_expected_n": float(exp_emg) if np.isfinite(exp_emg) else np.nan,
                "imu_expected_n": float(exp_imu) if np.isfinite(exp_imu) else np.nan,

                # modality availability
                "has_emg": bool(has_emg_win),
                "has_imu": bool(has_imu_win),

                # unique sample counts (for QC / features)
                "emg_n": int(emg_n),
                "imu_n": int(imu_n),

                # raw row counts (debug for long-format)
                "emg_n_rows": int(emg_n_rows),
                "imu_n_rows": int(imu_n_rows),

                # QC flags (keep NaN as NaN; DO NOT cast to bool)
                "emg_cov_ok": emg_cov_ok,
                "imu_cov_ok": imu_cov_ok,
                "emg_gap_ok": emg_gap_ok,
                "imu_gap_ok": imu_gap_ok,
                "emg_max_gap_s": float(emg_max_gap) if np.isfinite(emg_max_gap) else np.nan,
                "imu_max_gap_s": float(imu_max_gap) if np.isfinite(imu_max_gap) else np.nan,
                "emg_ok": emg_ok,
                "imu_ok": imu_ok,
            })

    windows_df = pd.DataFrame(rows)

    out_path = PHASE5_DIR / f"{subject_name}__windows.parquet"
    windows_df.to_parquet(out_path, index=False)

    print(f"\nPhase 5 completed for {subject_name}")
    print(f"  → windows: {out_path}")
    if windows_df.empty:
        print("  WARNING: windows_df is empty. Check episodes or win params.")
        return windows_df

    # Summary (NaNs are ignored by mean — exactly what we want for missing modalities)
    qc_cols = ["emg_cov_ok","emg_gap_ok","emg_ok","imu_cov_ok","imu_gap_ok","imu_ok"]
    print("\nWindow QC summary (mean; NaNs ignored):")
    display(windows_df[qc_cols].mean(numeric_only=True))

    # Extra: missing modality ratios
    print("\nMissing modality ratios:")
    print("  EMG missing ratio:", float((~windows_df["has_emg"]).mean()))
    print("  IMU missing ratio:", float((~windows_df["has_imu"]).mean()))

    print("\nWindows per trial (top 12):")
    display(
        windows_df.groupby(["trial_id","episode_id"])["window_id"].count()
                 .reset_index(name="n_windows")
                 .sort_values("n_windows", ascending=False)
                 .head(12)
    )

    # Optional QC plots
    if PLOT_QC:
        trials = windows_df["trial_id"].unique().tolist()[:MAX_TRIALS_TO_PLOT]
        print(f"\nPlotting QC for {len(trials)} trials (max={MAX_TRIALS_TO_PLOT}) ...")
        for tid in trials:
            ep_t = episodes[episodes["trial_id"] == tid]
            w_t  = windows_df[windows_df["trial_id"] == tid]
            plot_trial_windows(subject_name, tid, ep_t, w_t)

    return windows_df

# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_12"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)


In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")


In [ ]:
# =========================================
# Phase 5 - Segmentation (Time-based Windowing) — UPDATED (modality-aware + long-format safe)
# Builds per-window metadata aligned to Phase 4 episodes
# Output: phase_05_windows/{subject}__windows.parquet
# Last update: 2026-02-15
#
# Key updates:
#  1) Handles missing IMU / missing EMG per-window:
#     - emg_ok / imu_ok become NaN when that modality is missing in the window (instead of False)
#     - adds has_emg / has_imu flags
#  2) Long-format safe counting:
#     - emg_n / imu_n = number of UNIQUE timestamps in the window (not number of rows)
#     - adds emg_n_rows / imu_n_rows for debugging
#  3) Time-gap QC uses UNIQUE timestamps (avoids repeated timestamps from long format)
#  4) rows.append stores NaN as-is (no bool(...) casting that would corrupt NaNs)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2B_DIR     = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_NORM_DIR = PROCESSED_ROOT / "phase_03_normalization" / "normalized_env_per_subject"
PHASE4_DIR      = PROCESSED_ROOT / "phase_04_events"
PHASE5_DIR      = PROCESSED_ROOT / "phase_05_windows"

PHASE5_DIR.mkdir(parents=True, exist_ok=True)

print("Paths loaded.")
print("PHASE5_DIR:", PHASE5_DIR)

# ─── Config ────────────────────────────────────────
WIN_LEN_S = 0.200
OVERLAP = 0.50
STEP_S = WIN_LEN_S * (1.0 - OVERLAP)

# Window QC
MIN_COVERAGE_RATIO = 0.60       # >=60% of expected unique timestamps must exist
MAX_TIME_GAP_S = 0.050          # flag if max Δt inside window > 50ms (on unique timestamps)
INCLUDE_CONTEXT = False         # default: windows strictly within [t_on, t_off]
CONTEXT_PAD_S = 0.30            # only if INCLUDE_CONTEXT=True

# Minimal plotting / QC
PLOT_QC = True
MAX_TRIALS_TO_PLOT = 6

print(f"Windowing: win={WIN_LEN_S:.3f}s | overlap={OVERLAP:.2f} | step={STEP_S:.3f}s")
print(f"QC: MIN_COVERAGE_RATIO={MIN_COVERAGE_RATIO} | MAX_TIME_GAP_S={MAX_TIME_GAP_S}s")
print(f"Context: INCLUDE_CONTEXT={INCLUDE_CONTEXT} | CONTEXT_PAD_S={CONTEXT_PAD_S}s")

# ─── Helpers ───────────────────────────────────────
def estimate_fs(t: np.ndarray) -> float:
    """Robust sampling rate estimate from time vector (seconds)."""
    t = np.asarray(t, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 10:
        return np.nan
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 5:
        return np.nan
    return float(1.0 / np.median(dt))

def make_windows(t_on: float, t_off: float, win_len_s: float, step_s: float):
    """Windows fully inside [t_on, t_off]. Returns list of (t_start, t_end)."""
    if (t_off - t_on) < win_len_s:
        return []
    starts = np.arange(t_on, t_off - win_len_s + 1e-12, step_s)
    return [(float(s), float(s + win_len_s)) for s in starts]

def slice_by_time(df: pd.DataFrame, tcol: str, t_start: float, t_end: float) -> pd.DataFrame:
    """Inclusive start, exclusive end."""
    return df.loc[(df[tcol] >= t_start) & (df[tcol] < t_end)]

def window_timegap_ok(t_unique_sorted: np.ndarray, max_gap_s: float) -> tuple[bool, float]:
    """Check max gap on UNIQUE timestamps."""
    t = np.asarray(t_unique_sorted, dtype=float)
    t = t[np.isfinite(t)]
    if len(t) < 2:
        return False, np.nan
    gaps = np.diff(t)
    gaps = gaps[np.isfinite(gaps)]
    if len(gaps) == 0:
        return False, np.nan
    mg = float(np.max(gaps))
    return (mg <= max_gap_s), mg

def expected_samples(fs: float, win_len_s: float) -> float:
    if not np.isfinite(fs) or fs <= 0:
        return np.nan
    return float(fs * win_len_s)

def qc_coverage(n_samples_unique_time: int, n_expected: float, min_ratio: float) -> bool:
    """Coverage check using UNIQUE timestamp counts."""
    if not np.isfinite(n_expected) or n_expected <= 0:
        return n_samples_unique_time >= 5
    return (n_samples_unique_time >= min_ratio * n_expected)

# ─── QC Plot ───────────────────────────────────────
def plot_trial_windows(subject_name: str, trial_id: str, episodes_trial: pd.DataFrame, windows_trial: pd.DataFrame):
    """
    Plot IMU gyro_gate if available; otherwise plot EMG proxy (max env_norm_*).
    Overlay episodes and some window start lines.
    """
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"

    plt.figure(figsize=(14, 4))
    plotted = False

    # Try IMU gate first
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu = df_imu[(df_imu["subject"].astype(str) == str(subject_name)) &
                        (df_imu["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_imu.empty and "t_imu" in df_imu.columns and "gyro_gate" in df_imu.columns:
            df_imu = df_imu.sort_values("t_imu")
            plt.plot(df_imu["t_imu"], df_imu["gyro_gate"], lw=1.1, label="gyro_gate (IMU)")
            plotted = True

    # EMG fallback proxy if IMU not plotted
    if (not plotted) and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg = df_emg[(df_emg["subject"].astype(str) == str(subject_name)) &
                        (df_emg["trial_id"].astype(str) == str(trial_id))].copy()
        if not df_emg.empty and "t_emg" in df_emg.columns:
            df_emg = df_emg.sort_values("t_emg")
            env_cols = [c for c in df_emg.columns if c.startswith("env_norm_")]
            if env_cols:
                gate = df_emg[env_cols].max(axis=1)
                plt.plot(df_emg["t_emg"], gate, lw=1.1, label="max(env_norm_*) (EMG proxy)")
                plotted = True

    # Overlay episodes
    episodes_trial = episodes_trial.sort_values("t_on")
    for i, (_, r) in enumerate(episodes_trial.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.12, color="orange", label="episode" if i == 0 else None)

    # Overlay a few window starts (downsample to avoid clutter)
    wshow = windows_trial.sort_values("t_start").head(60)
    for _, r in wshow.iterrows():
        plt.axvline(r["t_start"], alpha=0.18, lw=0.8, color="black")

    title_extra = "" if plotted else " (no gate plotted)"
    plt.title(f"{subject_name} — {trial_id} | episodes={len(episodes_trial)} | windows={len(windows_trial)}{title_extra}")
    plt.xlabel("Time (s)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ─── Phase 5 runner ────────────────────────────────
def run_phase5_for_subject(subject_name: str):
    ep_path  = PHASE4_DIR / f"{subject_name}__episodes.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"

    if not ep_path.exists():
        raise FileNotFoundError(f"Episodes file not found: {ep_path}")

    episodes = pd.read_parquet(ep_path)
    episodes["subject"] = episodes["subject"].astype(str)
    episodes["trial_id"] = episodes["trial_id"].astype(str)

    df_emg = pd.read_parquet(emg_path) if emg_path.exists() else pd.DataFrame()
    df_imu = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()

    if not df_emg.empty:
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
        if "t_emg" not in df_emg.columns:
            raise ValueError(f"EMG file missing 't_emg': {emg_path}")

    if not df_imu.empty:
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
        if "t_imu" not in df_imu.columns:
            raise ValueError(f"IMU file missing 't_imu': {imu_path}")

    # Pre-estimate fs per trial using UNIQUE timestamps (robust to long-format)
    fs_emg_by_trial = {}
    fs_imu_by_trial = {}

    if not df_emg.empty:
        for tid, g in df_emg.groupby("trial_id"):
            fs_emg_by_trial[tid] = estimate_fs(np.sort(g["t_emg"].unique()))

    if not df_imu.empty:
        for tid, g in df_imu.groupby("trial_id"):
            fs_imu_by_trial[tid] = estimate_fs(np.sort(g["t_imu"].unique()))

    rows = []

    # Build windows per (trial, episode)
    for (trial_id, episode_id), ep_grp in episodes.groupby(["trial_id", "episode_id"]):
        ep0 = ep_grp.iloc[0]
        t_on  = float(ep0["t_on"])
        t_off = float(ep0["t_off"])

        # optional context
        if INCLUDE_CONTEXT:
            t_on_eff  = t_on  - CONTEXT_PAD_S
            t_off_eff = t_off + CONTEXT_PAD_S
        else:
            t_on_eff, t_off_eff = t_on, t_off

        win_list = make_windows(t_on_eff, t_off_eff, WIN_LEN_S, STEP_S)
        if not win_list:
            continue

        # Trial data (may be missing modality)
        emg_trial = pd.DataFrame()
        imu_trial = pd.DataFrame()

        if not df_emg.empty:
            emg_trial = df_emg[df_emg["trial_id"] == trial_id].sort_values("t_emg")
        if not df_imu.empty:
            imu_trial = df_imu[df_imu["trial_id"] == trial_id].sort_values("t_imu")

        fs_emg = fs_emg_by_trial.get(trial_id, np.nan)
        fs_imu = fs_imu_by_trial.get(trial_id, np.nan)
        exp_emg = expected_samples(fs_emg, WIN_LEN_S)
        exp_imu = expected_samples(fs_imu, WIN_LEN_S)

        for w_id, (ts, te) in enumerate(win_list, start=1):
            # Slice windows (can be empty if modality missing)
            emg_win = slice_by_time(emg_trial, "t_emg", ts, te) if not emg_trial.empty else pd.DataFrame()
            imu_win = slice_by_time(imu_trial, "t_imu", ts, te) if not imu_trial.empty else pd.DataFrame()

            has_emg_win = (len(emg_win) > 0)
            has_imu_win = (len(imu_win) > 0)

            # -------------------------
            # EMG QC (modality-aware)
            # -------------------------
            emg_n_rows = int(len(emg_win))
            emg_n = int(emg_win["t_emg"].nunique()) if emg_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_emg_win:
                emg_cov_ok  = np.nan
                emg_gap_ok  = np.nan
                emg_ok      = np.nan
                emg_max_gap = np.nan
            else:
                emg_cov_ok = qc_coverage(emg_n, exp_emg, MIN_COVERAGE_RATIO) if emg_n > 0 else False

                if emg_n > 1:
                    t_emg_u = np.sort(emg_win["t_emg"].unique())
                    emg_gap_ok, emg_max_gap = window_timegap_ok(t_emg_u, MAX_TIME_GAP_S)
                else:
                    emg_gap_ok, emg_max_gap = (False, np.nan)

                emg_ok = bool(emg_cov_ok and emg_gap_ok)

            # -------------------------
            # IMU QC (modality-aware)
            # -------------------------
            imu_n_rows = int(len(imu_win))
            imu_n = int(imu_win["t_imu"].nunique()) if imu_n_rows > 0 else 0  # UNIQUE timestamps

            if not has_imu_win:
                imu_cov_ok  = np.nan
                imu_gap_ok  = np.nan
                imu_ok      = np.nan
                imu_max_gap = np.nan
            else:
                imu_cov_ok = qc_coverage(imu_n, exp_imu, MIN_COVERAGE_RATIO) if imu_n > 0 else False

                if imu_n > 1:
                    t_imu_u = np.sort(imu_win["t_imu"].unique())
                    imu_gap_ok, imu_max_gap = window_timegap_ok(t_imu_u, MAX_TIME_GAP_S)
                else:
                    imu_gap_ok, imu_max_gap = (False, np.nan)

                imu_ok = bool(imu_cov_ok and imu_gap_ok)

            rows.append({
                "subject": subject_name,
                "trial_id": str(trial_id),
                "episode_id": int(episode_id),
                "window_id": int(w_id),

                "t_start": float(ts),
                "t_end": float(te),
                "win_len_s": float(WIN_LEN_S),
                "step_s": float(STEP_S),

                "task_type": ep0.get("task_type", "unknown"),
                "data_source": ep0.get("data_source", "unknown"),
                "gate_source": ep0.get("gate_source", "unknown"),

                "t_on": t_on,
                "t_off": t_off,
                "context_used": bool(INCLUDE_CONTEXT),

                "fs_emg_est": float(fs_emg) if np.isfinite(fs_emg) else np.nan,
                "fs_imu_est": float(fs_imu) if np.isfinite(fs_imu) else np.nan,
                "emg_expected_n": float(exp_emg) if np.isfinite(exp_emg) else np.nan,
                "imu_expected_n": float(exp_imu) if np.isfinite(exp_imu) else np.nan,

                # modality availability
                "has_emg": bool(has_emg_win),
                "has_imu": bool(has_imu_win),

                # unique sample counts (for QC / features)
                "emg_n": int(emg_n),
                "imu_n": int(imu_n),

                # raw row counts (debug for long-format)
                "emg_n_rows": int(emg_n_rows),
                "imu_n_rows": int(imu_n_rows),

                # QC flags (keep NaN as NaN; DO NOT cast to bool)
                "emg_cov_ok": emg_cov_ok,
                "imu_cov_ok": imu_cov_ok,
                "emg_gap_ok": emg_gap_ok,
                "imu_gap_ok": imu_gap_ok,
                "emg_max_gap_s": float(emg_max_gap) if np.isfinite(emg_max_gap) else np.nan,
                "imu_max_gap_s": float(imu_max_gap) if np.isfinite(imu_max_gap) else np.nan,
                "emg_ok": emg_ok,
                "imu_ok": imu_ok,
            })

    windows_df = pd.DataFrame(rows)

    out_path = PHASE5_DIR / f"{subject_name}__windows.parquet"
    windows_df.to_parquet(out_path, index=False)

    print(f"\nPhase 5 completed for {subject_name}")
    print(f"  → windows: {out_path}")
    if windows_df.empty:
        print("  WARNING: windows_df is empty. Check episodes or win params.")
        return windows_df

    # Summary (NaNs are ignored by mean — exactly what we want for missing modalities)
    qc_cols = ["emg_cov_ok","emg_gap_ok","emg_ok","imu_cov_ok","imu_gap_ok","imu_ok"]
    print("\nWindow QC summary (mean; NaNs ignored):")
    display(windows_df[qc_cols].mean(numeric_only=True))

    # Extra: missing modality ratios
    print("\nMissing modality ratios:")
    print("  EMG missing ratio:", float((~windows_df["has_emg"]).mean()))
    print("  IMU missing ratio:", float((~windows_df["has_imu"]).mean()))

    print("\nWindows per trial (top 12):")
    display(
        windows_df.groupby(["trial_id","episode_id"])["window_id"].count()
                 .reset_index(name="n_windows")
                 .sort_values("n_windows", ascending=False)
                 .head(12)
    )

    # Optional QC plots
    if PLOT_QC:
        trials = windows_df["trial_id"].unique().tolist()[:MAX_TRIALS_TO_PLOT]
        print(f"\nPlotting QC for {len(trials)} trials (max={MAX_TRIALS_TO_PLOT}) ...")
        for tid in trials:
            ep_t = episodes[episodes["trial_id"] == tid]
            w_t  = windows_df[windows_df["trial_id"] == tid]
            plot_trial_windows(subject_name, tid, ep_t, w_t)

    return windows_df

# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_13"   # ← change here

print(f"\n=== Starting Phase 5 for {SUBJECT_NAME} ===\n")
windows_df = run_phase5_for_subject(SUBJECT_NAME)


In [ ]:
# =========================================
# Phase 5 - QC Checker (OK/WARN/FAIL) + One Global CSV
# Run AFTER Phase 5 (windows_df is available)
# Output: phase_05_windows/phase5_qc_report.csv  (one file, all subjects)
# =========================================

import numpy as np
import pandas as pd

# --- Required globals ---
assert "windows_df" in globals(), "windows_df not found. Run Phase 5 first."
assert "SUBJECT_NAME" in globals(), "SUBJECT_NAME not found."
assert "PHASE5_DIR" in globals(), "PHASE5_DIR not found."
assert "WIN_LEN_S" in globals() and "STEP_S" in globals(), "WIN_LEN_S / STEP_S not found."

subject_name = str(SUBJECT_NAME)
w = windows_df.copy()

# --- Minimal required columns ---
req = ["trial_id","episode_id","window_id","t_start","t_end","t_on","t_off","win_len_s","step_s","emg_ok","imu_ok","fs_emg_est","fs_imu_est"]
missing = [c for c in req if c not in w.columns]
if missing:
    raise ValueError(f"windows_df missing columns: {missing}")

# --- Helpers ---
def ok_fail(condition: bool, ok_msg: str, fail_msg: str):
    return ("OK", ok_msg) if condition else ("FAIL", fail_msg)

def ok_warn_fail(value: float, ok_thr: float, warn_thr: float, name: str):
    """
    Larger-is-better (ratios). OK if >= ok_thr, WARN if >= warn_thr, else FAIL.
    """
    if not np.isfinite(value):
        return "FAIL", f"{name}: FAIL (NaN)"
    if value >= ok_thr:
        return "OK", f"{name}: OK ({value:.3f})"
    if value >= warn_thr:
        return "WARN", f"{name}: WARN ({value:.3f})"
    return "FAIL", f"{name}: FAIL ({value:.3f})"

def approx_equal_ratio(x: pd.Series, target: float, atol: float):
    return float(np.mean(np.isclose(x.to_numpy(dtype=float), target, atol=atol)))

def median_ape(observed: pd.Series, expected: pd.Series):
    o = observed.to_numpy(dtype=float)
    e = expected.to_numpy(dtype=float)
    m = np.isfinite(o) & np.isfinite(e) & (e > 0)
    if not np.any(m):
        return np.nan
    return float(np.median(np.abs(o[m] - e[m]) / e[m]))

# --- Compute checks ---
# 1) Non-empty
n_windows = int(len(w))
status_nonempty, msg_nonempty = ok_fail(n_windows > 0, "windows_df: OK (non-empty)", "windows_df: FAIL (empty)")

# 2) Window length correct
w["win_len_actual"] = w["t_end"] - w["t_start"]
win_len_ok_ratio = approx_equal_ratio(w["win_len_actual"], WIN_LEN_S, atol=1e-6)
st_winlen, msg_winlen = ok_warn_fail(win_len_ok_ratio, ok_thr=0.999, warn_thr=0.98, name="Window length match ratio")

# 3) Windows inside episode (strict)
inside_ratio = float(np.mean((w["t_start"] >= w["t_on"] - 1e-9) & (w["t_end"] <= w["t_off"] + 1e-9)))
st_inside, msg_inside = ok_warn_fail(inside_ratio, ok_thr=0.999, warn_thr=0.98, name="Inside-episode ratio")

# 4) Sampling rates sanity (expected approx 1259 & 148 in your dataset)
# Use wide tolerances to be safe across subjects
fs_emg_med = float(np.nanmedian(w["fs_emg_est"]))
fs_imu_med = float(np.nanmedian(w["fs_imu_est"]))

st_fs_emg, msg_fs_emg = ok_warn_fail(
    1.0 if (1100 <= fs_emg_med <= 1400) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_emg median in [1100,1400] (median={fs_emg_med:.1f})"
)
st_fs_imu, msg_fs_imu = ok_warn_fail(
    1.0 if (120 <= fs_imu_med <= 180) else 0.0,
    ok_thr=1.0, warn_thr=1.0,
    name=f"fs_imu median in [120,180] (median={fs_imu_med:.1f})"
)

# 5) EMG/IMU QC ratios
emg_ok_ratio = float(np.mean(w["emg_ok"].astype(bool))) if n_windows else np.nan
imu_ok_ratio = float(np.mean(w["imu_ok"].astype(bool))) if n_windows else np.nan
st_emgok, msg_emgok = ok_warn_fail(emg_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="EMG window OK ratio")
st_imuok, msg_imuok = ok_warn_fail(imu_ok_ratio, ok_thr=0.98, warn_thr=0.90, name="IMU window OK ratio")

# 6) Episode expected window count consistency (median abs diff should be ~0)
ep = (w.groupby(["trial_id","episode_id"])
        .agg(t_on=("t_on","first"), t_off=("t_off","first"), n_windows=("window_id","count"))
        .reset_index())
ep["dur_s"] = ep["t_off"] - ep["t_on"]
ep["n_expected"] = (np.floor((ep["dur_s"] - WIN_LEN_S) / STEP_S).clip(lower=0) + 1).astype(int)
ep["abs_diff"] = (ep["n_windows"] - ep["n_expected"]).abs()

ep_abs_diff_median = float(np.median(ep["abs_diff"])) if len(ep) else np.nan
# OK if median == 0, WARN if <=1, FAIL otherwise
if np.isfinite(ep_abs_diff_median):
    if ep_abs_diff_median == 0:
        st_ep, msg_ep = "OK",  "Episode window count: OK (median abs diff = 0)"
    elif ep_abs_diff_median <= 1:
        st_ep, msg_ep = "WARN","Episode window count: WARN (median abs diff <= 1)"
    else:
        st_ep, msg_ep = "FAIL",f"Episode window count: FAIL (median abs diff = {ep_abs_diff_median:.1f})"
else:
    st_ep, msg_ep = "FAIL", "Episode window count: FAIL (NaN)"

# 7) Optional: if you patched Phase 5 to store unique-time counts (recommended), check APE
# If not patched, this will likely FAIL/WARN because emg_n/imu_n are row counts (long-format).
ape_emg = np.nan
ape_imu = np.nan
st_ape_emg, msg_ape_emg = "SKIP", "EMG samples-vs-expected: SKIP (emg_n not present)"
st_ape_imu, msg_ape_imu = "SKIP", "IMU samples-vs-expected: SKIP (imu_n not present)"

if "emg_n" in w.columns and "imu_n" in w.columns:
    w["emg_expected"] = w["fs_emg_est"] * w["win_len_s"]
    w["imu_expected"] = w["fs_imu_est"] * w["win_len_s"]
    ape_emg = median_ape(w["emg_n"], w["emg_expected"])
    ape_imu = median_ape(w["imu_n"], w["imu_expected"])

    # If emg_n/imu_n are unique-time samples, APE should be small (e.g., < 5%)
    # If still row counts, APE will be huge, so it will WARN/FAIL (which is informative).
    def ape_status(ape, name):
        if not np.isfinite(ape):
            return "FAIL", f"{name}: FAIL (NaN)"
        if ape <= 0.05:
            return "OK",  f"{name}: OK (median APE={ape:.3f})"
        if ape <= 0.20:
            return "WARN",f"{name}: WARN (median APE={ape:.3f})"
        return "FAIL", f"{name}: FAIL (median APE={ape:.3f})"

    st_ape_emg, msg_ape_emg = ape_status(ape_emg, "EMG samples-vs-expected")
    st_ape_imu, msg_ape_imu = ape_status(ape_imu, "IMU samples-vs-expected")

# --- Overall status ---
statuses = [status_nonempty, st_winlen, st_inside, st_fs_emg, st_fs_imu, st_emgok, st_imuok, st_ep]
# treat WARN as pass, FAIL as fail
overall = "OK" if all(s != "FAIL" for s in statuses) else "FAIL"

# --- Print human-friendly report ---
print(f"\n=== Phase 5 QC Report | {subject_name} ===")
for s, m in [
    (status_nonempty, msg_nonempty),
    (st_winlen, msg_winlen),
    (st_inside, msg_inside),
    (st_fs_emg, msg_fs_emg),
    (st_fs_imu, msg_fs_imu),
    (st_emgok, msg_emgok),
    (st_imuok, msg_imuok),
    (st_ep, msg_ep),
    (st_ape_emg, msg_ape_emg),
    (st_ape_imu, msg_ape_imu),
]:
    print(f"[{s}] {m}")

print(f"Overall QC: {overall}")

# --- Save/update ONE global CSV (append or replace subject) ---
report_path = PHASE5_DIR / "phase5_qc_report.csv"

row = {
    "subject": subject_name,
    "overall": overall,

    # statuses (easy to read)
    "nonempty": status_nonempty,
    "win_len_match": st_winlen,
    "inside_episode": st_inside,
    "fs_emg_ok": st_fs_emg,
    "fs_imu_ok": st_fs_imu,
    "emg_ok_ratio_status": st_emgok,
    "imu_ok_ratio_status": st_imuok,
    "episode_count_status": st_ep,
    "emg_samples_vs_expected": st_ape_emg,
    "imu_samples_vs_expected": st_ape_imu,

    # minimal numbers (for debugging if something fails; not too many)
    "n_windows_total": n_windows,
    "n_trials": int(w["trial_id"].nunique()),
    "n_episodes": int(len(ep)),
    "fs_emg_median": fs_emg_med,
    "fs_imu_median": fs_imu_med,
    "emg_ok_ratio": emg_ok_ratio,
    "imu_ok_ratio": imu_ok_ratio,
    "episode_abs_diff_median": ep_abs_diff_median,
    "ape_emg_median": ape_emg,
    "ape_imu_median": ape_imu,
}

new_df = pd.DataFrame([row])

if report_path.exists():
    old = pd.read_csv(report_path)
    old = old[old["subject"].astype(str) != subject_name]  # replace on rerun
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(report_path, index=False)
print(f"\n✅ Saved/updated global QC CSV: {report_path}")
